In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

print("Getting Data from HuggingFace...")
url_ic = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ic_before_after.csv"
url_ec = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ec_before_after.csv"

df_ic = pd.read_csv(url_ic)
df_ec = pd.read_csv(url_ec)

ec_target_col = "after_style_mean"
df_ec["after_pct"] = df_ec[ec_target_col]
if df_ec["after_pct"].max() <= 10.0:
    df_ec["after_pct"] = df_ec["after_pct"] * 10

df_ic = df_ic[
    df_ic["model"].str.contains("lora", case=False)
    & ~df_ic["model"].str.contains("base", case=False)
].copy()
df_ec = df_ec[
    df_ec["model"].str.contains("lora", case=False)
    & ~df_ec["model"].str.contains("base", case=False)
].copy()

df_ic["Metric"] = "Intrinsic Capability (IC)"
df_ec["Metric"] = "Extrinsic Capability (EC)"
df_all = pd.concat([df_ic, df_ec], ignore_index=True)
print("Dataset is ready!")

Getting Data from HuggingFace...
Dataset is ready!


In [14]:
def get_architecture(name):
    name = name.lower()
    if "qwen" in name and "coder" in name:
        return "Qwen-Coder"
    elif "qwen" in name:
        return "Qwen (Base)"
    elif "llama" in name:
        return "Llama"
    elif "gemma" in name:
        return "Gemma"
    elif "phi" in name:
        return "Phi"
    return "Other"


def get_tier(name):
    name = name.lower()
    if any(s in name for s in ["7b", "8b", "9b", "big"]):
        return "Big"
    elif "qwen" in name and "coder" in name and "7b" not in name:
        return "Small"
    elif any(s in name for s in ["1.5b", "2b", "3b", "mini", "small"]):
        return "Small"
    return "Unknown"


df_all["Architecture"] = df_all["model"].apply(get_architecture)
df_all["Tier"] = df_all["model"].apply(get_tier)
df_all["Data Quality"] = df_all["model"].apply(
    lambda x: "High Quality (HQ)" if "high_quality" in x else "ALL Dataset"
)
df_all["Model Group"] = df_all["Architecture"] + " (" + df_all["Tier"] + ")"

exp_mapping = {
    "exp-103": {"dataset": "1K", "quantization": "4-bit"},
    "exp-104": {"dataset": "5K", "quantization": "4-bit"},
    "exp-105": {"dataset": "1K", "quantization": "8-bit"},
    "exp-106": {"dataset": "5K", "quantization": "8-bit"},
    "exp-107": {"dataset": "1K", "quantization": "16-bit"},
    "exp-108": {"dataset": "5K", "quantization": "16-bit"},
    "exp-109": {"dataset": "1K", "quantization": "4-bit"},
    "exp-110": {"dataset": "5K", "quantization": "4-bit"},
    "exp-111": {"dataset": "1K", "quantization": "8-bit"},
    "exp-112": {"dataset": "5K", "quantization": "8-bit"},
    "exp-113": {"dataset": "1K", "quantization": "16-bit"},
    "exp-114": {"dataset": "5K", "quantization": "16-bit"},
}

exp_df = (
    pd.DataFrame.from_dict(exp_mapping, orient="index")
    .reset_index()
    .rename(columns={"index": "experiment"})
)
df_all = pd.merge(df_all, exp_df, on="experiment", how="left")
df_all = df_all.dropna(subset=["quantization"])
df_all["quantization"] = pd.Categorical(
    df_all["quantization"], categories=["4-bit", "8-bit", "16-bit"], ordered=True
)
print("Experiment match is completed.")

Experiment match is completed.


In [15]:
os.makedirs("../images", exist_ok=True)


def plot_deep_dive(selected_group):
    df_filtered = df_all[df_all["Model Group"] == selected_group].copy()
    if df_filtered.empty:
        print(f"{selected_group} için veri bulunamadı.")
        return

    df_ic_plot = df_filtered[df_filtered["Metric"] == "Intrinsic Capability (IC)"]
    df_ec_plot = df_filtered[df_filtered["Metric"] == "Extrinsic Capability (EC)"]

    g = sns.catplot(
        data=df_ic_plot,
        x="quantization",
        y="after_pct",
        row="Data Quality",
        col="dataset",
        kind="bar",
        color="#3498db",
        edgecolor="black",
        height=5,
        aspect=1.2,
        zorder=1,
    )

    g.set(ylim=(0, 110))

    g.set_axis_labels(
        "Quantization Precision", "Relative Performance Scale (%)", fontweight="bold"
    )
    g.set_titles("Dataset: {col_name} | {row_name}", fontweight="bold", size=14)
    g.fig.suptitle(
        f"{selected_group} Deep Dive\n(Bars = IC: Logic Solved/161 | Line = EC: Format Score/10)",
        fontweight="bold",
        y=1.08,
        size=16,
    )

    row_names = g.row_names
    col_names = g.col_names

    for i, row in enumerate(row_names):
        for j, col in enumerate(col_names):
            ax = g.axes[i, j]

            for container in ax.containers:
                labels = []
                for bar in container:
                    h = bar.get_height()
                    if pd.isna(h) or h == 0:
                        labels.append("")
                    else:
                        raw_ic = int(round(h * 161 / 100))
                        labels.append(f"{raw_ic}")
                ax.bar_label(
                    container,
                    labels=labels,
                    padding=4,
                    fontsize=12,
                    fontweight="bold",
                    color="#2980b9",
                )

            subset_ec = df_ec_plot[
                (df_ec_plot["Data Quality"] == row) & (df_ec_plot["dataset"] == col)
            ].sort_values("quantization")

            if not subset_ec.empty:
                x_coords = range(len(subset_ec["quantization"]))
                y_coords = subset_ec["after_pct"].values

                ax.plot(
                    x_coords,
                    y_coords,
                    marker="o",
                    color="#e67e22",
                    linewidth=3,
                    markersize=10,
                    zorder=3,
                )

                for x, y in zip(x_coords, y_coords):
                    if not pd.isna(y):
                        raw_ec = y / 10
                        ax.annotate(
                            f"{raw_ec:.1f}",
                            (x, y),
                            textcoords="offset points",
                            xytext=(0, 15),
                            ha="center",
                            fontsize=12,
                            fontweight="bold",
                            color="#d35400",
                            bbox=dict(
                                boxstyle="round,pad=0.3",
                                fc="white",
                                ec="#e67e22",
                                alpha=0.9,
                            ),
                        )

    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    custom_lines = [
        Patch(
            facecolor="#3498db",
            edgecolor="black",
            label="Intrinsic (Logic/Questions Solved)",
        ),
        Line2D(
            [0],
            [0],
            color="#e67e22",
            lw=3,
            marker="o",
            markersize=8,
            label="Extrinsic (Syntax/Format Score)",
        ),
    ]
    g.fig.legend(
        handles=custom_lines,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.05),
        ncol=2,
        frameon=True,
    )

    sns.despine()
    (
        selected_group.replace(" ", "_").replace("(", "").replace(")", "").lower()
    )
    # plt.savefig(f'../images/deep_dive_combined_{safe_name}.png', dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

In [16]:
unique_groups = sorted(
    [
        g
        for g in df_all["Model Group"].unique()
        if "Other" not in g and "Unknown" not in g
    ]
)

dropdown = widgets.Dropdown(
    options=unique_groups,
    description="Choose a Model:",
    style={"description_width": "initial"},
)
out_plot = widgets.Output()


def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with out_plot:
            clear_output(wait=True)
            plot_deep_dive(change["new"])


dropdown.observe(on_change)
display(dropdown, out_plot)

with out_plot:
    plot_deep_dive(unique_groups[0])

Dropdown(description='Choose a Model:', options=('Gemma (Big)', 'Gemma (Small)', 'Llama (Big)', 'Llama (Small)…

Output()